# Modulo 1: Ingesta

Este cuaderno recorre el dataset, extrae metadatos por video y genera el manifiesto base.


## Librerias y parametros de ingesta


In [4]:
from pathlib import Path

import cv2
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ingesta"
VIDEO_MANIFEST_PATH = OUTPUT_DIR / "video_manifest.csv"

SPLITS = ["train", "val", "test"]
CLASS_NAMES = ["normal", "hurto_simulado"]

SKIP_INGESTA = VIDEO_MANIFEST_PATH.exists()

if SKIP_INGESTA:
    print(f"Manifiesto existente: {VIDEO_MANIFEST_PATH}")
    print("Cargando datos. Borra el archivo si deseas regenerar.")
    video_df = pd.read_csv(VIDEO_MANIFEST_PATH)
else:
    print(f"No existe {VIDEO_MANIFEST_PATH}. Se procesara el dataset completo.")

pd.Series(
    {
        "DATA_DIR": str(DATA_DIR),
        "OUTPUT_DIR": str(OUTPUT_DIR),
        "VIDEO_MANIFEST_PATH": str(VIDEO_MANIFEST_PATH),
        "SKIP_INGESTA": SKIP_INGESTA,
    }
)

Manifiesto existente: c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIFICIAL\PROJECT\ENTREGA_3\outputs\ingesta\video_manifest.csv
Cargando datos. Borra el archivo si deseas regenerar.


DATA_DIR               c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...
OUTPUT_DIR             c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...
VIDEO_MANIFEST_PATH    c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...
SKIP_INGESTA                                                        True
dtype: object

## Videos encontrados en el dataset


In [5]:
if not SKIP_INGESTA:

    video_rows = []

    for split_name in SPLITS:
        for class_name in CLASS_NAMES:
            class_dir = DATA_DIR / split_name / class_name
            for video_path in sorted(class_dir.glob("*.mp4")):
                video_rows.append(
                    {
                        "split": split_name,
                        "class_name": class_name,
                        "video_path_absolute": str(video_path.resolve()),
                        "video_path": video_path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix(),
                    }
                )

    video_items_df = pd.DataFrame(video_rows).sort_values(["split", "class_name", "video_path"]).reset_index(drop=True)

    print(f"Videos descubiertos: {len(video_items_df)}")
    display(video_items_df.groupby(["split", "class_name"]).size().reset_index(name="num_videos"))
    display(video_items_df.head())

else:
    print("Usando datos existentes. Saltando descubrimiento de videos.")

Usando datos existentes. Saltando descubrimiento de videos.


## Metadatos por video


In [6]:
if not SKIP_INGESTA:

    video_metadata_rows = []

    for _, row in video_items_df.iterrows():
        split_name = str(row["split"])
        class_name = str(row["class_name"])
        video_path_absolute = Path(str(row["video_path_absolute"]))

        capture = cv2.VideoCapture(str(video_path_absolute))
        if not capture.isOpened():
            raise RuntimeError(f"No se pudo abrir el video: {video_path_absolute}")

        fps_original = float(capture.get(cv2.CAP_PROP_FPS))
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        capture.release()

        if fps_original <= 0:
            raise ValueError(f"FPS invalido para el video: {video_path_absolute}")

        duration_seconds = frame_count / fps_original
        video_metadata_rows.append(
            {
                "split": split_name,
                "class_name": class_name,
                "video_path": str(row["video_path"]),
                "video_id": f"{split_name}__{class_name}__{video_path_absolute.stem}",
                "fps_original": round(fps_original, 6),
                "frame_count": int(frame_count),
                "duration_seconds": round(duration_seconds, 6),
                "width": int(width),
                "height": int(height),
            }
        )

    video_df = pd.DataFrame(video_metadata_rows).sort_values(["split", "class_name", "video_id"]).reset_index(drop=True)

    print(f"Videos con metadatos: {len(video_df)}")
    display(
        video_df.groupby(["split", "class_name"])
        .agg(
            num_videos=("video_id", "size"),
            duracion_media_s=("duration_seconds", "mean"),
            frames_promedio=("frame_count", "mean"),
        )
        .round(2)
        .reset_index()
    )

else:
    print(f"Usando datos existentes ({len(video_df)} videos). Saltando extraccion de metadatos.")

Usando datos existentes (328 videos). Saltando extraccion de metadatos.


## Guardado del manifiesto


In [7]:
if not SKIP_INGESTA:

    video_df[
        [
            "split",
            "class_name",
            "video_path",
            "video_id",
            "fps_original",
            "frame_count",
            "duration_seconds",
            "width",
            "height",
        ]
    ].to_csv(VIDEO_MANIFEST_PATH, index=False)

    print(VIDEO_MANIFEST_PATH)
    print(f"Videos procesados: {len(video_df)}")
else:
    print(f"Manifiesto reutilizado: {VIDEO_MANIFEST_PATH}")
    print(f"Videos en manifiesto: {len(video_df)}")

Manifiesto reutilizado: c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIFICIAL\PROJECT\ENTREGA_3\outputs\ingesta\video_manifest.csv
Videos en manifiesto: 328


## Verificacion del manifiesto


In [8]:
saved_video_manifest_df = pd.read_csv(VIDEO_MANIFEST_PATH)

display(
    pd.DataFrame(
        [
            {
                "artifacto": "video_manifest.csv",
                "filas": len(saved_video_manifest_df),
                "columnas": saved_video_manifest_df.shape[1],
                "videos_unicos": saved_video_manifest_df["video_id"].nunique(),
            },
        ]
    )
)

print("Videos por split y clase")
display(saved_video_manifest_df.groupby(["split", "class_name"]).size().reset_index(name="count"))

null_summary_df = pd.DataFrame(
    {
        "video_manifest_nulls": saved_video_manifest_df.isna().sum(),
    }
).fillna("")

display(null_summary_df)

,artifacto,filas,columnas,videos_unicos
0,video_manifest.csv,328,9,328


Videos por split y clase


,split,class_name,count
0,test,hurto_simulado,17
1,test,normal,33
2,train,hurto_simulado,76
3,train,normal,153
4,val,hurto_simulado,16
5,val,normal,33


,video_manifest_nulls
split,0
class_name,0
video_path,0
video_id,0
fps_original,0
frame_count,0
duration_seconds,0
width,0
height,0
